In [0]:
import requests
import json
from datetime import datetime, timezone

# ============================================================
# CONFIG
# ============================================================

POPULATION_API_URL = (
    "https://api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_5"
    "&drilldowns=Nation,Year"
    "&measures=Population"
)

RAW_POPULATION_PATH = (
    "/Volumes/bls_dataquest/bronze/bls_dq/raw/population/"
)

POPULATION_FILE_PATH = (
    RAW_POPULATION_PATH + "population.json"
)

HEADERS = {
    "User-Agent": "Databricks-Rearc-Productivity-Project/1.0"
}


# ============================================================
# CREATE RAW DIRECTORY
# ============================================================

dbutils.fs.mkdirs(
    RAW_POPULATION_PATH
)


# ============================================================
# FETCH LATEST POPULATION DATA
# ============================================================

response = requests.get(
    POPULATION_API_URL,
    headers=HEADERS,
    timeout=60
)

response.raise_for_status()

population_json = response.json()


# ============================================================
# BASIC VALIDATION
# ============================================================

if "data" not in population_json:
    raise ValueError(
        "API response does not contain 'data'."
    )

records = population_json["data"]

if not records:
    raise ValueError(
        "Population API returned zero records."
    )


# ============================================================
# SAVE LATEST RAW RESPONSE
# ============================================================

with open(
    POPULATION_FILE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        population_json,
        f,
        indent=2
    )


# ============================================================
# VERIFICATION
# ============================================================

years = sorted(
    int(record["Year"])
    for record in records
    if record.get("Year") is not None
)

print("=" * 60)
print("POPULATION INGESTION COMPLETED")
print("=" * 60)
print(f"HTTP Status : {response.status_code}")
print(f"Records     : {len(records)}")
print(f"Year Range  : {min(years)} - {max(years)}")
print(f"Output      : {POPULATION_FILE_PATH}")
print("=" * 60)

POPULATION INGESTION COMPLETED
HTTP Status : 200
Records     : 12
Year Range  : 2013 - 2024
Output      : /Volumes/bls_dataquest/bronze/bls_dq/raw/population/population.json
